# Temporal feature engineering

Add calendar variables and past-only same-zone rolling means to the 2024 lag dataset.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_with_lags.parquet"))


In [ ]:
X = df.copy()

X["is_weekend"] = (X["tpep_pickup_datetime"].dt.day_of_week >= 5).astype(int) #1 : weekend, 0 : weekdays

X["roll_mean_3"] = X.groupby("PULocationID")["trip_count"].transform(lambda s: s.shift(1).rolling(3).mean())
X["roll_mean_6"] = X.groupby("PULocationID")["trip_count"].transform(lambda s: s.shift(1).rolling(6).mean())
X["roll_mean_24"] = X.groupby("PULocationID")["trip_count"].transform(lambda s: s.shift(1).rolling(24).mean())

X["hours"] = X["tpep_pickup_datetime"].dt.hour
X["day_of_the_week"] = X["tpep_pickup_datetime"].dt.day_of_week
X["month"] = X["tpep_pickup_datetime"].dt.month


In [ ]:
from pathlib import Path

feature_data_dir = Path("data") if Path("data").is_dir() else Path("../data")
feature_output_path = feature_data_dir / "yellow_taxi_2024_features.parquet"
X.to_parquet(feature_output_path, index = False)
print(f"Saved {len(X):,} rows with engineered features to {feature_output_path}")
